In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
from scipy.stats import linregress
import mpmath
import matplotlib.pyplot as plt
import time
from numba import njit

# ================== 1. 真实的黎曼零点准备 ==================
mpmath.mp.dps = 15
N_ZEROS = 100
TRUE_ZEROS = np.array([float(mpmath.zetazero(i).imag) for i in range(1, N_ZEROS + 1)])

# ================== 2. 全概率溅射演化引擎 (Ulam's Exact) ==================
@njit
def run_universe_exact_splatting(steps, n_bins, u_c, k_opt, c_offset):
    transitions = np.zeros((n_bins, n_bins), dtype=np.float64)
    V = np.zeros(n_bins, dtype=np.float64)
    
    # 物理空间划分：严格均分 [-1.0, 1.0]
    dx = 2.0 / n_bins
    
    # 宇宙起始点 x = 0.5 所在格子
    init_bin = int((0.5 + 1.0) / dx)
    if init_bin >= n_bins: init_bin = n_bins - 1
    V[init_bin] = 1.0 # 初始时刻，概率为 1 的冲激函数
    
    for n in range(1, steps + 1):
        # 冷却退火公式（从右向左降温）
        mu_raw = u_c + k_opt / (np.log(n + c_offset)**2)
        
        # 物理保险丝
        if mu_raw > 2.0: mu = 2.0
        elif mu_raw < 0.1: mu = 0.1
        else: mu = mu_raw
            
        V_next = np.zeros(n_bins, dtype=np.float64)
        
        # 遍历所有存在概率的格子（上帝视角的波函数演化）
        for i in range(n_bins):
            if V[i] < 1e-12: # 算力优化：忽略空寂的宇宙区域
                continue
                
            e_left = -1.0 + i * dx
            e_right = e_left + dx
            
            y1 = 1.0 - mu * e_left * e_left
            y2 = 1.0 - mu * e_right * e_right
            
            # 处理抛物线顶点 (x=0) 包含在当前格子的情况
            if e_left <= 0.0 and e_right >= 0.0:
                y_max = 1.0
                y_min = min(y1, y2)
            else:
                y_max = max(y1, y2)
                y_min = min(y1, y2)
                
            if y_max > 1.0: y_max = 1.0
            if y_min < -1.0: y_min = -1.0
                
            w = y_max - y_min
            
            # 寻找溅射的目标格子范围
            j_min = int((y_min + 1.0) / dx)
            j_max = int((y_max + 1.0) / dx)
            if j_min < 0: j_min = 0
            if j_max >= n_bins: j_max = n_bins - 1
            
            # 核心逻辑：溅射范围与格子大小的精确映射
            if w <= 1e-12:
                # 极度收缩区，概率全部砸入单一目标格子
                j_center = int(((y_min + y_max)/2.0 + 1.0) / dx)
                if j_center < 0: j_center = 0
                if j_center >= n_bins: j_center = n_bins - 1
                
                flow = V[i]
                V_next[j_center] += flow
                transitions[i, j_center] += flow
            else:
                # 根据重叠面积（Overlap），将概率精确溅射分配
                for j in range(j_min, j_max + 1):
                    E_left = -1.0 + j * dx
                    E_right = E_left + dx
                    
                    overlap_left = max(y_min, E_left)
                    overlap_right = min(y_max, E_right)
                    overlap = overlap_right - overlap_left
                    
                    if overlap > 0.0:
                        prob = overlap / w
                        flow = V[i] * prob
                        V_next[j] += flow
                        transitions[i, j] += flow
                        
        V = V_next # 波函数推进到下一时刻
        
    return transitions

# ================== 3. 参数空间 500点 超清扫描设置 ==================
# 替换为 500 个高清探测点 (覆盖从深层周期区到最大热寂区)
test_points_end = np.linspace(1.20, 2.00, 500)

# ⚠️ 考虑到全概率计算极其耗时，设为 10 万步以保证能在合理时间内出图
# 如果你的机器算力极强且有耐心，可以改回 10**6 甚至 10**7
TOTAL_STEPS = 10**5 
C_OFFSET = 10.0   
DELTA_MU_ABS = 0.02  # 游标卡尺宽度：退火冷却 0.02

results_R2 = []
results_mean_err = []

print(f"🚀 启动【上帝视角全概率溅射】500点地毯式扫描 | 冷却 Δμ = -{DELTA_MU_ABS}\n")
print(f"{'进度':<10} | {'结束 μ (冷)':<12} | {'R²':<8} | {'平均误差':<10} | {'耗时 (s)'}")
print("-" * 65)

start_total_t = time.time()

for idx, mu_end in enumerate(test_points_end):
    start_point_t = time.time()
    
    # 逆向推导渐近线 U_C 和 K
    t_start_val = 1.0 / (np.log(1 + C_OFFSET)**2)
    t_end_val   = 1.0 / (np.log(TOTAL_STEPS + C_OFFSET)**2)
    k_opt = DELTA_MU_ABS / (t_start_val - t_end_val)
    u_c = mu_end - k_opt * t_end_val
    
    # 注入全概率演化引擎
    trans = run_universe_exact_splatting(TOTAL_STEPS, 5000, u_c, k_opt, C_OFFSET)
    
    # 构造概率转移矩阵并归一化
    P_sparse = sp.csr_matrix(trans, dtype=np.float64)
    row_sums = np.array(P_sparse.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1.0 
    P_sparse.data /= row_sums[P_sparse.indices]
    
    try:
        # 提取复数特征值
        eigenvalues, _ = eigs(P_sparse, k=N_ZEROS + 20, which='LM', tol=1e-4)
        phases = np.sort(np.angle(eigenvalues[np.abs(eigenvalues.imag) > 1e-4]))
        unwrapped = np.unwrap(phases)
        
        min_len = min(len(unwrapped), N_ZEROS)
        
        if min_len > 10:
            unwrapped_trunc = unwrapped[:min_len]
            true_zeros_trunc = TRUE_ZEROS[:min_len]
            
            slope, intercept, r_val, _, _ = linregress(unwrapped_trunc, true_zeros_trunc)
            pred = slope * unwrapped_trunc + intercept
            err = np.mean(np.abs(pred - true_zeros_trunc))
            r2 = r_val**2
        else:
            err = 20.0 
            r2 = 0.0
            
    except Exception:
        err = 20.0
        r2 = 0.0
        
    results_R2.append(r2)
    results_mean_err.append(err)
    
    # 实时打印进度 (每 20 个点汇报一次)
    if (idx + 1) % 20 == 0 or idx == 0:
        elapsed = time.time() - start_point_t
        print(f"[{idx+1:>3d}/500]  | {mu_end:<12.4f} | {r2:<8.4f} | {err:<10.4f} | {elapsed:.2f}s")

print("-" * 65)
print(f"✅ 上帝视角波函数扫描完成！总耗时: {(time.time()-start_total_t)/60:.2f} 分钟")

# ================== 4. 高清双轴绘图环节 ==================
fig, ax1 = plt.subplots(figsize=(15, 7.5))

ax1_twin = ax1.twinx()

# 绘制 R² (蓝色) 和 平均误差 (红色)
ax1.plot(test_points_end, results_R2, 'b-', lw=1.5, alpha=0.9, label='Macro $R^2$ (Universality)')
ax1.fill_between(test_points_end, results_R2, color='blue', alpha=0.08)
ax1_twin.plot(test_points_end, results_mean_err, 'r-', lw=1.2, alpha=0.75, label='Micro Mean Error (Precision)')

# 标注神圣物理坐标点
ax1.axvline(1.543689, color='k', linestyle='--', lw=2.5, alpha=0.8, label='Band-Merging ($1.5437$)')
ax1.axvline(1.401155, color='orange', linestyle=':', lw=2.5, alpha=0.8, label='Feigenbaum Point ($1.4011$)')
ax1.axvline(1.25, color='purple', linestyle='-.', lw=1.5, alpha=0.5, label='Period-2 to 4 Bifurcation ($\sim 1.25$)')
ax1.axvline(1.7548, color='green', linestyle=':', lw=1.5, alpha=0.5, label='Period-3 Window ($1.7548$)')

# 图表装饰与标签
ax1.set_xlabel(r'Cooling Endpoint Parameter ($\mu_{end}$)', fontsize=15, fontweight='bold')
ax1.set_ylabel(r'Riemann Homomorphism ($R^2$)', color='b', fontsize=15, fontweight='bold')
ax1_twin.set_ylabel('Mean Absolute Error', color='r', fontsize=15, fontweight='bold')
ax1.set_title(f'Exact Probability Splatting (500 Points): Cooling $\Delta\mu = -{DELTA_MU_ABS}$', fontsize=17)

# 动态参数说明悬浮窗
text_str = f"Ulam's Exact Method:\nSteps = $10^5$\nCooling Width = {DELTA_MU_ABS}\nBins = 5000"
ax1.text(0.02, 0.85, text_str, transform=ax1.transAxes, fontsize=12,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))

# 图例与网格
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right', bbox_to_anchor=(0.98, 0.45), fontsize=11)

plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

<>:185: SyntaxWarning: invalid escape sequence '\s'
<>:192: SyntaxWarning: invalid escape sequence '\D'
<>:185: SyntaxWarning: invalid escape sequence '\s'
<>:192: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_2956/4041733575.py:185: SyntaxWarning: invalid escape sequence '\s'
  ax1.axvline(1.25, color='purple', linestyle='-.', lw=1.5, alpha=0.5, label='Period-2 to 4 Bifurcation ($\sim 1.25$)')
/tmp/ipykernel_2956/4041733575.py:192: SyntaxWarning: invalid escape sequence '\D'
  ax1.set_title(f'Exact Probability Splatting (500 Points): Cooling $\Delta\mu = -{DELTA_MU_ABS}$', fontsize=17)


🚀 启动【上帝视角全概率溅射】500点地毯式扫描 | 冷却 Δμ = -0.02

进度         | 结束 μ (冷)     | R²       | 平均误差       | 耗时 (s)
-----------------------------------------------------------------
[  1/500]  | 1.2000       | 0.9888   | 5.1371     | 1.71s
[ 20/500]  | 1.2305       | 0.9877   | 5.4336     | 0.87s
[ 40/500]  | 1.2625       | 0.9680   | 8.8963     | 0.84s
[ 60/500]  | 1.2946       | 0.9912   | 4.5455     | 0.88s
[ 80/500]  | 1.3267       | 0.9872   | 5.3547     | 0.87s
[100/500]  | 1.3587       | 0.9730   | 7.7105     | 0.85s
[120/500]  | 1.3908       | 0.9844   | 5.9082     | 0.86s
